In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display
from sklearn.decomposition import PCA
from statsmodels.stats.outliers_influence import variance_inflation_factor

from analysis.src.io import experiment_paths, load_data
from analysis.src.constants import COMPONENTS, COVARIATES

In [ ]:
OUT = experiment_paths('index/structure')

data = load_data()

In [ ]:
# Summarise neighbourhood characteristics and correlations
numeric = data.select_dtypes(include = np.number)
numeric.describe(percentiles = [0.25, 0.5, 0.75]).T.rename_axis('variable').reset_index().to_csv(
    OUT.descriptives_csv, index = False
)

correlation = data[COMPONENTS].corr()
correlation.to_csv(OUT.component_correlations_csv, index_label = 'component')

contextual = data[[*COMPONENTS, *COVARIATES, 'z_nox', 'z_ndvi']].corr()
contextual.to_csv(OUT.contextual_correlations_csv, index_label = 'variable')

In [ ]:
# Examine component collinearity and principal components
component_data = data[COMPONENTS].dropna()
design = sm.add_constant(component_data)
vifs = {
    column: variance_inflation_factor(design.values, i) for i, column in enumerate(design.columns) if column != 'const'
}

pca = PCA(n_components = 5).fit(component_data)
loadings = pd.DataFrame(pca.components_.T, index = COMPONENTS, columns = [f'PC{i + 1}' for i in range(5)])

if loadings.loc[['amenity_score', 'dwelling_score', 'street_score', 'transport_score'], 'PC1'].mean() < 0:
    loadings['PC1'] *= -1

structure = loadings.copy()
structure['vif'] = pd.Series(vifs)

for name, column in [('nox_r', 'z_nox'), ('ndvi_r', 'z_ndvi'), ('imd_r', 'z_imd_score')]:
    structure[name] = contextual.loc[COMPONENTS, column]

structure.rename_axis('component').reset_index().to_csv(OUT.index_structure_csv, index = False)

pd.DataFrame({'component': loadings.columns, 'explained_variance_ratio': pca.explained_variance_ratio_}).to_csv(
    OUT.pca_variance_csv, index = False
)

display(structure)
display(correlation)